In [2]:
import pandas as pd
import numpy as np

from google.colab import files

uploaded = files.upload()

# load dataset
df = pd.read_csv("energy_prices_clean.csv")

# date
df['Date'] = pd.to_datetime(df['Date'])

# sort
df = df.sort_values('Date')

# returns
df['Brent_Return'] = df['Brent_Price'].pct_change()
df['TTF_Return'] = df['TTF_Price'].pct_change()
df['EUA_Return'] = df['EUA_Price'].pct_change()
df['Power_Return'] = df['Power_Index'].pct_change()

print(df.head())

Saving energy_prices_clean.csv to energy_prices_clean.csv
        Date  Brent_Price  TTF_Price  EUA_Price  Power_Index  Brent_Return  \
0 2022-01-04    80.000000  88.740997  84.800003    23.230589           NaN   
1 2022-01-05    80.800003  91.522003  87.519997    23.329651      0.010000   
2 2022-01-06    81.989998  96.501999  86.815002    22.606482      0.014728   
3 2022-01-07    81.750000  88.174004  85.300003    22.378633     -0.002927   
4 2022-01-10    80.870003  84.607002  80.419998    21.843687     -0.010764   

   TTF_Return  EUA_Return  Power_Return  
0         NaN         NaN           NaN  
1    0.031338    0.032075      0.004264  
2    0.054413   -0.008055     -0.030998  
3   -0.086299   -0.017451     -0.010079  
4   -0.040454   -0.057210     -0.023904  


In [3]:
#Rolling Volatility
df['Brent_30D_Vol'] = (
    df['Brent_Return']
    .rolling(30)
    .std()
)

df['TTF_30D_Vol'] = (
    df['TTF_Return']
    .rolling(30)
    .std()
)

df['EUA_30D_Vol'] = (
    df['EUA_Return']
    .rolling(30)
    .std()
)

df['Power_30D_Vol'] = (
    df['Power_Return']
    .rolling(30)
    .std()
)

In [4]:
#Historical VaR
brent_var = df['Brent_Return'].quantile(0.05)
ttf_var = df['TTF_Return'].quantile(0.05)
eua_var = df['EUA_Return'].quantile(0.05)
power_var = df['Power_Return'].quantile(0.05)

df['Brent_VaR95'] = brent_var
df['TTF_VaR95'] = ttf_var
df['EUA_VaR95'] = eua_var
df['Power_VaR95'] = power_var

In [5]:
#Rolling Correlations
df['Brent_TTF_Corr'] = (
    df['Brent_Return']
    .rolling(30)
    .corr(df['TTF_Return'])
)

df['TTF_Power_Corr'] = (
    df['TTF_Return']
    .rolling(30)
    .corr(df['Power_Return'])
)

df['EUA_Power_Corr'] = (
    df['EUA_Return']
    .rolling(30)
    .corr(df['Power_Return'])
)

In [6]:
#Sharpe Ratios
df['Brent_Sharpe'] = (
    df['Brent_Return']
    .rolling(252)
    .mean()
    /
    df['Brent_Return']
    .rolling(252)
    .std()
)

df['TTF_Sharpe'] = (
    df['TTF_Return']
    .rolling(252)
    .mean()
    /
    df['TTF_Return']
    .rolling(252)
    .std()
)

df['EUA_Sharpe'] = (
    df['EUA_Return']
    .rolling(252)
    .mean()
    /
    df['EUA_Return']
    .rolling(252)
    .std()
)

df['Power_Sharpe'] = (
    df['Power_Return']
    .rolling(252)
    .mean()
    /
    df['Power_Return']
    .rolling(252)
    .std()
)

In [7]:
#Drawdowns
for asset in ['Brent','TTF','EUA','Power']:

    cumulative = (
        1 + df[f'{asset}_Return']
    ).cumprod()

    running_max = cumulative.cummax()

    df[f'{asset}_Drawdown'] = (
        cumulative /
        running_max
        - 1
    )

In [8]:
df.to_csv(
    'energy_prices_quant.csv',
    index=False
)

print(df.columns)
print(df.shape)

Index(['Date', 'Brent_Price', 'TTF_Price', 'EUA_Price', 'Power_Index',
       'Brent_Return', 'TTF_Return', 'EUA_Return', 'Power_Return',
       'Brent_30D_Vol', 'TTF_30D_Vol', 'EUA_30D_Vol', 'Power_30D_Vol',
       'Brent_VaR95', 'TTF_VaR95', 'EUA_VaR95', 'Power_VaR95',
       'Brent_TTF_Corr', 'TTF_Power_Corr', 'EUA_Power_Corr', 'Brent_Sharpe',
       'TTF_Sharpe', 'EUA_Sharpe', 'Power_Sharpe', 'Brent_Drawdown',
       'TTF_Drawdown', 'EUA_Drawdown', 'Power_Drawdown'],
      dtype='object')
(979, 28)
